In [1]:
import numpy as np
import matplotlib.pyplot as plt
from py_wake_ellipsys.wind_farm_models.ellipsys import EllipSys
from pyellipsys.inversemap import InverseMap
import xarray as xr
import pandas as pd
import os
os.chdir('/work3/s234229/bach_repo/RANS')
from py_wake_postprocessor import process_rans

In [2]:
file = os.path.abspath('../../nc files/flowdata_02m_8m.nc')
ds = xr.open_dataset(file)

In [6]:
ds_cart = process_rans(file, res=4.0, size=(512, 512, 100), filename='../../nc files/flowdata_z02m_8m_mb_cartesian.nc')

Target grid size: (129, 129, 100)
Total points per variable: 1,664,100
Interpolating dataset onto the new grid...
Saving to ../../nc files/flowdata_z02m_8m_mb_cartesian.nc...
Processing complete!


In [5]:
import os
import numpy as np
import xarray as xr

def process_rans(file, res=10.0, size=(512, 512, 100), filename=None):
    """
    Post-process RANS data to create a cartesian grid netcdf file.
    Parameters:
    - file: Path to the input RANS netcdf file.
    - res: Resolution in meters for the new x-y plane (default: 10.0).
    - size: Tuple specifying the size in each direction of x,y of the cartesian grid 
            and size from 0 of z (default: (512,512,100)).
        - z will be logarithmically spaced from 0.1 to 100 meters with size[2] points.
    - filename: Optional name for the output cartesian netcdf file. If None, it will be generated from the input filename.
    """
    if filename is None:
        filename = os.path.splitext(os.path.basename(file))[0] + '_cartesian.nc'

    # 1. Open the dataset
    ds = xr.open_dataset(file)

    # Create new cartesian grid based on the bounds and resolution
    x_vec = np.arange(-size[0]//2, size[0]//2+res, res)
    y_vec = np.arange(-size[1]//2, size[1]//2+res, res)
    z_vec = np.logspace(-1, 2, num=size[2], base=10)  # Logarithmic spacing in z to capture near-ground details

    # 3. Guardrail check for memory usage
    total_points = len(x_vec) * len(y_vec) * len(z_vec)
    print(f"Target grid size: ({len(x_vec)}, {len(y_vec)}, {len(z_vec)})")
    print(f"Total points per variable: {total_points:,}")
    
    if total_points > 10_000_000:
        ans = input(f"The target grid has {total_points:,} points, which may lead to long processing times and high memory usage. Do you want to proceed? (y/n): ")
        if ans.lower() != 'y':
            print("Aborting processing.")
            return None

    # 4. Perform the 3D interpolation
    # 'linear' is fast and safe. You can use 'cubic' if you need smoother gradients.
    print("Interpolating dataset onto the new grid...")
    ds_cartesian = ds.interp(
        x=x_vec, 
        y=y_vec, 
        z=z_vec, 
        method='linear',
        kwargs={"fill_value": "extrapolate"} # Extrapolates if your target grid slightly exceeds bounds
    )

    # 5. Save and clean up
    print(f"Saving to {filename}...")
    ds_cartesian.to_netcdf(filename)
    print("Processing complete!")
    
    return ds_cartesian